In [7]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os

def random_subsampling(points, n_samples):
    start = time.time()
    indices = np.random.choice(points.shape[0], n_samples, replace=False)
    return points[indices], time.time() - start

def voxel_grid_subsampling(points, voxel_size):
    start = time.time()
    coords = (points / voxel_size).astype(int)
    _, inv = np.unique(coords, axis=0, return_index=True)
    return points[inv], time.time() - start

def farthest_point_sampling(points, n_samples):
    start = time.time()
    farthest_pts = np.zeros((n_samples, 3))
    farthest_pts[0] = points[np.random.randint(len(points))]
    distances = np.linalg.norm(points - farthest_pts[0], axis=1)
    for i in range(1, n_samples):
        idx = np.argmax(distances)
        farthest_pts[i] = points[idx]
        distances = np.minimum(distances, np.linalg.norm(points - farthest_pts[i], axis=1))
    return farthest_pts, time.time() - start

def run_suite(file_path):
    if not os.path.exists(file_path):
        return
        
    raw_data = np.loadtxt(file_path)
    points = raw_data[:, :3]
    
    res_r, t_r = random_subsampling(points, 15000)
    res_v, t_v = voxel_grid_subsampling(points, 0.3)
    res_f, t_f = farthest_point_sampling(points, 1500)
    
    np.savetxt('subsampled_random.xyz', res_r)
    np.savetxt('subsampled_voxel.xyz', res_v)
    np.savetxt('subsampled_fps.xyz', res_f)
    
    fig = plt.figure(figsize=(18, 5))
    data_list = [('Original', points), ('Random', res_r), ('Voxel', res_v), ('FPS', res_f)]
    for i, (name, p) in enumerate(data_list):
        ax = fig.add_subplot(1, 4, i+1, projection='3d')
        ax.scatter(p[:,0], p[:,1], p[:,2], s=0.5, c=p[:,2], cmap='viridis')
        ax.set_title(name)
    plt.savefig('subsampling_comparison.png')
    plt.close()

if __name__ == "__main__":
    run_suite('terra_02_000004.asc')